# Linear Regression từ Scratch

Xây dựng toàn bộ Linear Regression **không dùng** `nn.Linear` hay `optim` — tự viết từng bước để hiểu rõ cơ chế bên trong.

In [1]:
import torch

true_w = torch.tensor([2, -3.4])
true_b = 4.2
n = 1000
X = torch.randn(n, 2)

/home/sakana/miniconda3/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Cell 1 — Tạo dữ liệu giả (Synthetic Data)

Mục đích: ta biết **trước** $w$ và $b$ đúng để sau khi train kiểm tra xem model học có gần đúng không.

- `true_w = [2, -3.4]` — trọng số "chuẩn" cần học
- `true_b = 4.2` — bias "chuẩn"
- `X = torch.randn(1000, 2)` — 1000 điểm dữ liệu, mỗi điểm có 2 features, lấy từ phân phối chuẩn $\mathcal{N}(0, 1)$

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


## Cell 2 — Kiểm tra thiết bị (Device Check)

Nếu có GPU (CUDA) thì dùng GPU để tăng tốc, không thì dùng CPU.

> **Lưu ý:** Hiện tại máy phát hiện **RTX 2050** nhưng đang dùng driver `nouveau` (open-source) thay vì driver chính thức của NVIDIA → CUDA không hoạt động.  
> **Cách fix:** `sudo pacman -S nvidia nvidia-utils` rồi reboot.

In [4]:
noise = 0.01 * torch.randn(n, 1)

y = X @ true_w.reshape(-1, 1) + true_b + noise

## Cell 3 — Tạo nhãn y (Label Generation)

$$y = Xw + b + \epsilon, \quad \epsilon \sim \mathcal{N}(0,\, 0.01^2)$$

- `X @ true_w.reshape(-1, 1)`: **matmul** — shape `(1000,2) × (2,1) = (1000,1)`, trộn 2 features theo trọng số
- `true_w.reshape(-1, 1)`: đổi từ `(2,)` → `(2,1)` để chiều khớp với matmul
- `noise = 0.01 * torch.randn(n, 1)`: nhiễu nhỏ từ phân phối chuẩn → **giả lập sai số đo lường thực tế**

**Tại sao cần noise?**  
Không có noise, dữ liệu là đường thẳng tuyệt đối — không tồn tại trong thực tế.  
Có noise → model phải học **bất chấp nhiễu**, giống như bài toán thực. Độ lớn `0.01` rất nhỏ nên model vẫn hội tụ về gần đúng $w = [2, -3.4]$.

In [6]:
# tham so can hoc
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

## Cell 4 — Khởi tạo tham số (Parameter Initialization)

- `torch.normal(0, 0.01, size=(2,1))`: khởi tạo $w$ ngẫu nhiên gần 0, tránh gradient bùng nổ sớm
- `torch.zeros(1)`: khởi tạo $b = 0$
- `requires_grad=True`: yêu cầu PyTorch **theo dõi gradient** cho $w$ và $b$ — bắt buộc để `.backward()` hoạt động

> Nếu thiếu `requires_grad=True`, gọi `.backward()` sẽ báo lỗi vì PyTorch không có gì để differentiate.

In [7]:
# define model
def net(X):
    return X @ w + b


def loss(y_hat, y):
    return ((y_hat - y) ** 2 / 2).mean()


def sgd(params, lr):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad
            param.grad.zero_()
            


## Cell 5 — Định nghĩa Model, Loss, Optimizer

### `net(X)` — Forward pass
$$\hat{y} = Xw + b$$
Là **affine transformation** thủ công (tương đương `nn.Linear` nhưng tự viết).

### `loss(y_hat, y)` — Hàm mất mát MSE/2
$$\mathcal{L} = \frac{1}{n}\sum \frac{(\hat{y}_i - y_i)^2}{2}$$
- `(y_hat - y) ** 2`: **Hadamard** (bình phương từng phần tử, không trộn gì)
- Chia `2` để gradient gọn hơn: $\frac{d}{d\hat{y}}\frac{(\hat{y}-y)^2}{2} = (\hat{y}-y)$ thay vì $2(\hat{y}-y)$

### `sgd(params, lr)` — Stochastic Gradient Descent
```
param -= lr * param.grad   →  w = w - α·∇w
param.grad.zero_()         →  reset gradient, tránh cộng dồn sang vòng sau
```
- `torch.no_grad()`: tắt autograd khi cập nhật tham số — phép cập nhật **không phải bước học**, không muốn PyTorch track thêm
- `param.grad.zero_()`: in-place reset — nếu quên bước này, gradient sẽ cộng dồn qua nhiều batch → sai hoàn toàn

In [10]:
def data_iter(batch_size, X, y):
    idx = torch.randperm(X.shape[0])
    for i in range(0, X.shape[0], batch_size):
        indices = idx[i : i + batch_size]
        yield X[indices], y[indices]


lr = 0.03
batch_size = 32
epochs = 5

## Cell 6 — Data Iterator + Hyperparameters

```python
idx = torch.randperm(X.shape[0])   # shuffle: tạo permutation ngẫu nhiên của [0..999]
indices = idx[i : i + batch_size]   # lấy batch_size index liên tiếp từ permutation
yield X[indices], y[indices]        # trả từng batch (generator, không load hết RAM)
```

- `yield` → **generator**: chỉ tạo batch khi cần, tiết kiệm bộ nhớ
- Shuffle trước khi chia batch → tránh model học theo thứ tự dữ liệu (bias)

**Hyperparameters:**
| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `lr` | 0.03 | Tốc độ học — bước nhảy mỗi lần cập nhật |
| `batch_size` | 32 | Số mẫu mỗi mini-batch |
| `epochs` | 5 | Số lần duyệt toàn bộ dataset |

In [11]:
for epoch in range(epochs):
    for Xb, yb in data_iter(batch_size, X, y):
        l = loss(net(Xb), yb)
        l.backward()
        sgd([w, b], lr)
    with torch.no_grad():
        train_l = loss(net(X), y)
        print(f"epoch {epoch + 1}, loss {train_l:f}")


epoch 1, loss 0.013367
epoch 2, loss 0.002273
epoch 3, loss 0.000426
epoch 4, loss 0.000111
epoch 5, loss 0.000058


## Cell 7 — Training Loop

Luồng chuẩn mỗi iteration:

$$\underbrace{X \rightarrow \hat{y}}_{\text{forward}} \rightarrow \underbrace{\mathcal{L}}_{\text{loss}} \rightarrow \underbrace{\nabla w, \nabla b}_{\text{backward}} \rightarrow \underbrace{w, b \leftarrow w - \alpha\nabla w}_{\text{update}}$$

```python
l = loss(net(Xb), yb)   # 1. FORWARD: tính y_hat và loss
l.backward()             # 2. BACKWARD: autograd tính ∂L/∂w và ∂L/∂b
sgd([w, b], lr)          # 3. UPDATE: w = w - lr*grad, rồi zero_grad
```

Sau mỗi epoch, tính loss trên **toàn bộ** train set (không shuffle, không update) để theo dõi tiến trình hội tụ.  
`with torch.no_grad()`: tắt autograd để tiết kiệm bộ nhớ khi chỉ đánh giá.

In [12]:
print(f"True w: {true_w}, learned w: {w.reshape(-1)}")
print(f"w hoc duoc: {w.reshape(-1).detach()}")
print(f"True b: {true_b}, learned b: {b.item()}")

True w: tensor([ 2.0000, -3.4000]), learned w: tensor([ 1.9987, -3.3964], grad_fn=<ViewBackward0>)
w hoc duoc: tensor([ 1.9987, -3.3964])
True b: 4.2, learned b: 4.196910381317139


## Cell 8 — Đánh giá kết quả

So sánh tham số đã học với giá trị đúng ban đầu:

| | `w[0]` | `w[1]` | `b` |
|---|---|---|---|
| **True** | 2.0 | -3.4 | 4.2 |
| **Learned** | ~2.0 | ~-3.4 | ~4.2 |

- `.detach()`: tách tensor khỏi computation graph (chỉ lấy giá trị, không track gradient)
- `.item()`: chuyển tensor 1 phần tử thành số Python thuần (`float`)
- `.reshape(-1)`: ép về 1D để in đẹp hơn (từ `(2,1)` → `(2,)`)